# Etapa 3 — Baseline: FLAN-T5-base vs Qwen2.5-1.5B-Instruct (sin fine-tuning)

Evalúa ambos modelos sobre el test set (42 ejemplos) usando el prompt completo
(`construir_prompt()`) y calcula métricas automáticas.



## Celda 1 — Instalación de dependencias

In [ ]:


!pip install transformers torch accelerate sentencepiece rouge-score nltk -q

## Celda 2 — Configuración de entorno (Colab vs local)

In [ ]:
import os
import sys
from pathlib import Path

# Detectar si estamos en Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Montar Google Drive y apuntar al proyecto
    from google.colab import drive
    drive.mount('/content/drive')
    # Ajustar esta ruta al lugar donde tengas el proyecto en Drive
    PROJECT_ROOT = Path('/content/drive/MyDrive/inegi')
    sys.path.insert(0, str(PROJECT_ROOT))
else:
    # Local: subir un nivel desde notebooks/
    PROJECT_ROOT = Path('..').resolve()
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'Modo: {"Colab" if IN_COLAB else "Local"}')

## Celda 3 — Imports

In [ ]:
import json
import re
import csv
import torch
import pandas as pd
import nltk
from transformers import (
    T5ForConditionalGeneration, T5Tokenizer,
    AutoModelForCausalLM, AutoTokenizer,
)
from rouge_score import rouge_scorer

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Celda 4 — Cargar test set

In [ ]:
TEST_PATH = PROJECT_ROOT / 'training_data' / 'test_questions.json'
test = json.loads(TEST_PATH.read_text(encoding='utf-8'))

print(f'Ejemplos de test: {len(test)}')
print(f'Campos: {list(test[0].keys())}')
print(f'Intents únicos: {sorted(set(e["intent"] for e in test))}')

## Celda 5 — Funciones de inferencia

In [ ]:
def inferir_flan(model, tokenizer, prompt: str, max_new_tokens: int = 512) -> str:
    """Inferencia seq2seq para FLAN-T5."""
    inputs = tokenizer(
        prompt, return_tensors='pt', truncation=True, max_length=512
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(out[0], skip_special_tokens=True)


def inferir_qwen(model, tokenizer, prompt: str, max_new_tokens: int = 512) -> str:
    """Inferencia causal con chat template para Qwen2.5-Instruct."""
    messages = [{'role': 'user', 'content': prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    return tokenizer.decode(
        out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
    )

## Celda 6 — Funciones de métricas

In [ ]:
# Secciones esperadas en la estructura de respuesta
_SECCIONES = [
    r'resumen',
    r'datos\s+principales',
    r'interpretaci[oó]n',
    r'limitaciones',
]

# Patrones que indican posible número inventado (solo aplica si la respuesta
# no viene del gold; se usa como proxy, no es exacto)
_NUM_PATRON = re.compile(r'\b\d{4,}\b')  # números de 4+ dígitos
_NO_CAUSAL = re.compile(
    r'correlaci[oó]n|asociaci[oó]n|no\s+impl[ií]ca\s+causalidad|coincidencia\s+temporal',
    re.IGNORECASE
)


def _tiene_secciones(texto: str) -> float:
    """Fracción de secciones esperadas presentes (0.0–1.0)."""
    texto_lower = texto.lower()
    encontradas = sum(
        1 for s in _SECCIONES if re.search(s, texto_lower)
    )
    return encontradas / len(_SECCIONES)


def _menciona_no_causalidad(texto: str, intent: str) -> bool | None:
    """True si menciona no-causalidad (solo relevante para intent de cruce)."""
    if intent != 'relacion_ventas_accidentes_nacional':
        return None  # no aplica
    return bool(_NO_CAUSAL.search(texto))


def calcular_metricas_rouge(
    predicciones: list[str], referencias: list[str]
) -> dict:
    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'], use_stemmer=False
    )
    r1, r2, rl = [], [], []
    for pred, ref in zip(predicciones, referencias):
        s = scorer.score(ref, pred)
        r1.append(s['rouge1'].fmeasure)
        r2.append(s['rouge2'].fmeasure)
        rl.append(s['rougeL'].fmeasure)
    return {
        'rouge1': round(sum(r1) / len(r1), 4),
        'rouge2': round(sum(r2) / len(r2), 4),
        'rougeL': round(sum(rl) / len(rl), 4),
    }


def calcular_metricas_extra(
    predicciones: list[str],
    ejemplos: list[dict],
) -> dict:
    """Métricas adicionales: secciones, longitud, no-causalidad."""
    frac_secciones = [_tiene_secciones(p) for p in predicciones]
    longitudes = [len(p.split()) for p in predicciones]
    no_causal_results = [
        _menciona_no_causalidad(p, e['intent'])
        for p, e in zip(predicciones, ejemplos)
    ]
    # Solo promediar los intents donde aplica
    no_causal_vals = [v for v in no_causal_results if v is not None]
    return {
        'secciones_presentes': round(sum(frac_secciones) / len(frac_secciones), 4),
        'longitud_media_palabras': round(sum(longitudes) / len(longitudes), 1),
        'no_causalidad_cuando_aplica': (
            round(sum(no_causal_vals) / len(no_causal_vals), 4)
            if no_causal_vals else None
        ),
    }


print('Funciones de métricas listas.')

## Celda 7 — FLAN-T5-base: cargar y evaluar

> FLAN-T5-base (~250 MB) funciona en CPU y GPU. El prompt se trunca a 512 tokens
> (límite del modelo). Las respuestas serán cortas comparadas con el gold.

In [ ]:
FLAN_ID = 'google/flan-t5-base'

print(f'Cargando {FLAN_ID}...')
flan_tokenizer = T5Tokenizer.from_pretrained(FLAN_ID)
flan_model = T5ForConditionalGeneration.from_pretrained(FLAN_ID).to(device)
flan_model.eval()
print('Modelo cargado.')

In [ ]:
from tqdm.auto import tqdm

flan_preds = []
for ex in tqdm(test, desc='FLAN-T5'):
    pred = inferir_flan(flan_model, flan_tokenizer, ex['input'])
    flan_preds.append(pred)

print(f'Inferencia completada: {len(flan_preds)} predicciones.')

In [ ]:
# Liberar GPU antes de cargar Qwen
del flan_model
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print('FLAN-T5 descargado de memoria.')

## Celda 8 — Qwen2.5-1.5B-Instruct: cargar y evaluar

> ~3 GB en bfloat16. Requiere GPU con al menos 4 GB de VRAM (T4 Colab o RTX 3060 Ti local).
> Si hay OOM, reducir `max_new_tokens` a 256.

In [ ]:
QWEN_ID = 'Qwen/Qwen2.5-1.5B-Instruct'

print(f'Cargando {QWEN_ID}...')
qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_ID, trust_remote_code=True)
qwen_model = AutoModelForCausalLM.from_pretrained(
    QWEN_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
)
qwen_model.eval()
print('Modelo cargado.')

In [ ]:
qwen_preds = []
for ex in tqdm(test, desc='Qwen2.5-1.5B'):
    pred = inferir_qwen(qwen_model, qwen_tokenizer, ex['input'])
    qwen_preds.append(pred)

print(f'Inferencia completada: {len(qwen_preds)} predicciones.')

In [ ]:
del qwen_model
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print('Qwen descargado de memoria.')

## Celda 9 — Calcular métricas

In [ ]:
gold = [ex['output'] for ex in test]

flan_rouge = calcular_metricas_rouge(flan_preds, gold)
flan_extra = calcular_metricas_extra(flan_preds, test)

qwen_rouge = calcular_metricas_rouge(qwen_preds, gold)
qwen_extra = calcular_metricas_extra(qwen_preds, test)

print('Métricas FLAN-T5-base:', {**flan_rouge, **flan_extra})
print('Métricas Qwen2.5-1.5B:', {**qwen_rouge, **qwen_extra})

## Celda 10 — Tabla comparativa

In [ ]:
resultados = pd.DataFrame([
    {
        'Modelo': 'FLAN-T5-base (vanilla)',
        'ROUGE-1': flan_rouge['rouge1'],
        'ROUGE-2': flan_rouge['rouge2'],
        'ROUGE-L': flan_rouge['rougeL'],
        'Secciones (0-1)': flan_extra['secciones_presentes'],
        'Long. media (palabras)': flan_extra['longitud_media_palabras'],
        'No-causalidad (cruce)': flan_extra['no_causalidad_cuando_aplica'],
    },
    {
        'Modelo': 'Qwen2.5-1.5B-Instruct (vanilla)',
        'ROUGE-1': qwen_rouge['rouge1'],
        'ROUGE-2': qwen_rouge['rouge2'],
        'ROUGE-L': qwen_rouge['rougeL'],
        'Secciones (0-1)': qwen_extra['secciones_presentes'],
        'Long. media (palabras)': qwen_extra['longitud_media_palabras'],
        'No-causalidad (cruce)': qwen_extra['no_causalidad_cuando_aplica'],
    },
])

display(resultados.set_index('Modelo'))

## Celda 11 — Ejemplos cualitativos (3 ejemplos por intent)

In [ ]:
# Mostrar 3 ejemplos cubriendo intents distintos
INTENTS_MUESTRA = [
    'ranking_accidentes',
    'accidentes_por_tipo',
    'relacion_ventas_accidentes_nacional',
]

for intent_objetivo in INTENTS_MUESTRA:
    idx = next(
        (i for i, e in enumerate(test) if e['intent'] == intent_objetivo), None
    )
    if idx is None:
        print(f'Intent {intent_objetivo!r} no encontrado en test set.')
        continue

    print('=' * 70)
    print(f'INTENT: {intent_objetivo}')
    print('--- Gold (referencia) ---')
    print(gold[idx][:600])
    print('--- FLAN-T5-base ---')
    print(flan_preds[idx][:600])
    print('--- Qwen2.5-1.5B ---')
    print(qwen_preds[idx][:600])
    print()


## Celda 12 — Guardar resultados

Guarda `reports/etapa3_baseline_resultados.md` y `reports/baseline_results.csv`.

In [ ]:
REPORTS_DIR = PROJECT_ROOT / 'reports'
REPORTS_DIR.mkdir(exist_ok=True)

# --- CSV con predicciones completas ---
csv_path = REPORTS_DIR / 'baseline_results.csv'
rows = []
for i, ex in enumerate(test):
    rows.append({
        'idx': i,
        'intent': ex['intent'],
        'gold': ex['output'],
        'flan_pred': flan_preds[i],
        'qwen_pred': qwen_preds[i],
    })
pd.DataFrame(rows).to_csv(csv_path, index=False, encoding='utf-8')
print(f'CSV guardado: {csv_path}')

# --- Markdown con métricas ---
md_path = REPORTS_DIR / 'etapa3_baseline_resultados.md'

no_causal_flan = flan_extra['no_causalidad_cuando_aplica']
no_causal_qwen = qwen_extra['no_causalidad_cuando_aplica']

md = f"""# Etapa 3 — Resultados Baseline

## Configuración

- Test set: `training_data/test_questions.json` (42 ejemplos)
- Modelos evaluados sin fine-tuning (pesos pre-entrenados).
- ROUGE calculado con `use_stemmer=False` (español).

## Tabla de métricas

| Modelo | ROUGE-1 | ROUGE-2 | ROUGE-L | Secciones (0-1) | Long. media (pal.) | No-causalidad (cruce) |
|---|---|---|---|---|---|---|
| FLAN-T5-base (vanilla) | {flan_rouge['rouge1']} | {flan_rouge['rouge2']} | {flan_rouge['rougeL']} | {flan_extra['secciones_presentes']} | {flan_extra['longitud_media_palabras']} | {no_causal_flan} |
| Qwen2.5-1.5B-Instruct (vanilla) | {qwen_rouge['rouge1']} | {qwen_rouge['rouge2']} | {qwen_rouge['rougeL']} | {qwen_extra['secciones_presentes']} | {qwen_extra['longitud_media_palabras']} | {no_causal_qwen} |

## Notas

- **Secciones (0-1):** fracción de las 4 secciones esperadas (Resumen, Datos principales, Interpretación, Limitaciones) presentes en la respuesta.
- **No-causalidad (cruce):** fracción de ejemplos de `relacion_ventas_accidentes_nacional` donde la respuesta menciona no-causalidad/correlación.
- ROUGE subestima la calidad de respuestas largas y en español; usar como proxy comparativo, no como métrica absoluta.
- Qwen2.5-1.5B-Instruct (vanilla) debería superar a FLAN-T5-base por arquitectura y tamaño.
- Estos resultados son el baseline contra el que se comparará el modelo fine-tuneado de Etapa 4.

## Predicciones completas

Ver `reports/baseline_results.csv`.
"""

md_path.write_text(md, encoding='utf-8')
print(f'Markdown guardado: {md_path}')

---
**Etapa 3 completada.** Los archivos de salida están en `reports/`.

Siguiente: **Etapa 4** — Fine-tuning con LoRA/QLoRA en Colab (`04_finetune_qwen_lora_colab.ipynb`).